# Revisió manual de les respostes

Executa les cel·les en ordre amb el kernel de la `.venv` del projecte. Aquest notebook llegeix `results.json` sense tornar a cridar l'agent.

- **Good**: la resposta és correcta i està sustentada per les receptes recuperades.
- **Bad**: inventa informació, contradiu les receptes, no respon adequadament o falla.

Consulta les eines recuperades per comprovar la resposta. Pots afegir una nota abans de prémer un botó. Cada etiqueta es desa automàticament a `labels.json`, separat de `results.json`. En tornar a executar el notebook, recuperaràs la revisió pendent. Pots navegar i corregir etiquetes anteriors.

La pregunta de tipus `context-dependent` no porta historial al lot actual: tingues-ho en compte en jutjar-la.

In [1]:
import hashlib
import html
import json
from datetime import datetime, timezone
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

# Funciona obrint el notebook des de Week6 o des de l'arrel del repositori.
BASE_DIR = Path.cwd() if Path("results.json").is_file() else Path.cwd() / "Week6"
RESULTS_PATH = BASE_DIR / "results.json"
LABELS_PATH = BASE_DIR / "labels.json"

results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
if not isinstance(results, list) or not results:
    raise ValueError("results.json ha de contenir una llista no buida de resultats.")
if any(not isinstance(row, dict) or "question" not in row or "output" not in row
       for row in results):
    raise ValueError("Cada resultat ha de tenir question i output.")

# Evita aplicar etiquetes antigues a respostes d'una altra execució.
source_hash = hashlib.sha256(
    json.dumps(results, sort_keys=True, ensure_ascii=False).encode("utf-8")
).hexdigest()
reviews = {}
if LABELS_PATH.exists():
    saved = json.loads(LABELS_PATH.read_text(encoding="utf-8"))
    if saved.get("source_sha256") != source_hash:
        raise ValueError(
            "results.json ha canviat. Tria un altre LABELS_PATH o arxiva labels.json "
            "abans de començar una revisió nova."
        )
    for review in saved["reviews"]:
        if (type(review["index"]) is not int
                or not 0 <= review["index"] < len(results)
                or review["label"] not in {"good", "bad"}):
            raise ValueError("labels.json conté una etiqueta o un índex invàlid.")
        reviews[review["index"]] = review

print(f"{len(results)} resultats · {len(reviews)} revisats · Etiquetes: {LABELS_PATH.resolve()}")

20 resultats · 0 revisats · Etiquetes: /workspaces/ai-engineering-buildcamp-codespaces/Week6/labels.json


In [2]:
class ResultReviewer:
    def __init__(self, results, reviews, labels_path, source_hash):
        self.results = results
        self.reviews = dict(reviews)
        self.labels_path = labels_path
        self.source_hash = source_hash
        self.index = next((i for i in range(len(results)) if i not in reviews), 0)

        self.progress = widgets.IntProgress(max=len(results), description="Revisats:")
        self.position = widgets.HTML()
        self.content = widgets.HTML()
        self.context = widgets.HTML()
        self.details = widgets.Accordion(children=[self.context], selected_index=None)
        self.details.set_title(0, "Eines i context recuperat")
        self.notes = widgets.Textarea(
            description="Nota:", placeholder="Motiu de l'etiqueta (opcional)",
            layout=widgets.Layout(width="100%", height="85px"),
        )
        self.good = widgets.Button(description="Good", button_style="success")
        self.bad = widgets.Button(description="Bad", button_style="danger")
        self.previous = widgets.Button(description="Anterior")
        self.following = widgets.Button(description="Següent")
        self.notice = widgets.HTML()

        self.good.on_click(lambda _: self.label("good"))
        self.bad.on_click(lambda _: self.label("bad"))
        self.previous.on_click(lambda _: self.move(-1))
        self.following.on_click(lambda _: self.move(1))
        self.ui = widgets.VBox([
            self.progress, self.position, self.content, self.details, self.notes,
            widgets.HBox([self.good, self.bad, self.previous, self.following]),
            self.notice,
        ])
        self.render()

    def render(self):
        row = self.results[self.index]
        review = self.reviews.get(self.index, {})
        escape = lambda value: html.escape(str(value))
        self.progress.value = len(self.reviews)
        self.position.value = (
            f"<b>Escenari {self.index + 1}/{len(self.results)}</b> · "
            f"{len(self.reviews)} revisats · Etiqueta: {escape(review.get('label', 'pendent'))}"
        )
        answer = row.get("output")
        if answer is None:
            answer = f"Sense resposta. Error: {row.get('error', 'desconegut')}"
        self.content.value = (
            f"<p><b>Pregunta:</b> {escape(row['question'])}</p>"
            f"<p>Categoria: {escape(row.get('category', ''))} · "
            f"Tipus: {escape(row.get('type', ''))} · "
            f"Estat: {escape(row.get('status', ''))} · "
            f"Cost estimat (USD): {escape(row.get('cost'))}</p>"
            f"<pre style='white-space:pre-wrap'>{escape(answer)}</pre>"
        )
        self.context.value = (
            "<pre style='white-space:pre-wrap;max-height:450px;overflow:auto'>"
            + escape(json.dumps(row.get("tools", []), indent=2, ensure_ascii=False))
            + "</pre>"
        )
        self.notes.value = review.get("notes", "")
        self.previous.disabled = self.index == 0
        self.following.disabled = self.index == len(self.results) - 1

    def move(self, offset):
        self.index = max(0, min(len(self.results) - 1, self.index + offset))
        self.notice.value = ""
        self.render()

    def label(self, label):
        if label not in {"good", "bad"}:
            raise ValueError("L'etiqueta ha de ser good o bad.")
        row = self.results[self.index]
        updated = dict(self.reviews)
        updated[self.index] = {
            "index": self.index,
            "question": row["question"],
            "output": row["output"],
            "category": row.get("category"),
            "type": row.get("type"),
            "label": label,
            "notes": self.notes.value,
            "reviewed_at": datetime.now(timezone.utc).isoformat(),
        }
        payload = {
            "source_sha256": self.source_hash,
            "reviews": [updated[i] for i in sorted(updated)],
        }
        # Substitució atòmica: un error d'escriptura no deixa el JSON a mig fer.
        temporary = self.labels_path.with_suffix(".json.tmp")
        try:
            temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
            temporary.replace(self.labels_path)
        except OSError as exc:
            self.notice.value = f"<b>No s'ha pogut desar:</b> {html.escape(str(exc))}"
            return
        self.reviews = updated
        pending = [i for i in range(len(self.results)) if i not in self.reviews]
        if pending:
            self.index = next((i for i in pending if i > self.index), pending[0])
        self.render()
        self.notice.value = (
            "Etiqueta desada. "
            + ("Revisió completada! Pots corregir etiquetes amb Anterior/Següent."
               if not pending else "Següent escenari pendent.")
        )


reviewer = ResultReviewer(results, reviews, LABELS_PATH, source_hash)
display(reviewer.ui)

Les notes només es desen en prémer **Good** o **Bad**. Els botons **Anterior** i **Següent** permeten navegar sense etiquetar. Si corregeixes una etiqueta, es substitueix la revisió anterior del mateix escenari.

La cel·la següent llegeix les etiquetes desades i mostra el recompte actual. Torna-la a executar per actualitzar-lo.

In [4]:
if LABELS_PATH.exists():
    saved_reviews = json.loads(LABELS_PATH.read_text(encoding="utf-8"))["reviews"]
    for label in ("good", "bad"):
        print(f"{label}: {sum(row['label'] == label for row in saved_reviews)}")
    print(f"Pendents: {len(results) - len(saved_reviews)}")
else:
    print("Encara no hi ha etiquetes desades.")

good: 16
bad: 4
Pendents: 0
